# GemmaJudge â€” Gemma on AMD Instinctâ„¢ MI300X (proof-of-compute)

**Run this inside the AMD AI Notebooks Jupyter environment** (ROCm 7.2 + vLLM 0.16.0 + PyTorch 2.9).

It self-hosts **Gemma via vLLM on MI300X** and captures the artifacts that satisfy the
Track-3 requirement *"AMD compute usage is required; projects that do not demonstrate it
will be disqualified."* Gemma runs in its **load-bearing dual role** â€” the same open-weight
family generates the adversarial attacks **and** judges the answers â€” entirely on AMD.

### âš  Before you start
1. **Budget is 4 hrs / 24 hrs.** Don't leave this running. The last cell shuts the servers down.
2. **Gemma weights are gated on Hugging Face.** Accept the license for your chosen model at
   huggingface.co, create a read token, and paste it into the `HF_TOKEN` cell below.
3. **Screenshot** the `rocm-smi` cell output and the final ASR output â€” those go in the deck.

### What it produces (written to `amd_proof_artifacts/`)
`rocm_smi.txt` Â· `versions.txt` Â· `vllm_engine.log` (+ `vllm_target.log`) Â·
`eval_result.json` Â· `notes.md` Â· `serve_command.txt`. Copy these into the repo's
`docs/amd_proof/` and commit â€” that closes the disqualification gate.


In [ ]:
# --- Cell 1: hardware + environment proof (SCREENSHOT THIS CELL) ---
import subprocess, pathlib, datetime

OUT = pathlib.Path("amd_proof_artifacts"); OUT.mkdir(exist_ok=True)

def sh(cmd, tail=6000):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (("\n[stderr]\n" + r.stderr) if r.stderr else "")
    print(out[-tail:])
    return out

rocm = sh("rocm-smi")
(OUT / "rocm_smi.txt").write_text(rocm)

versions = sh('python - <<"PY"\n'
              'import torch, vllm\n'
              'print("torch", torch.__version__)\n'
              'print("vllm", vllm.__version__)\n'
              'print("rocm/hip", torch.version.hip)\n'
              'print("device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a")\n'
              'PY')
(OUT / "versions.txt").write_text(versions)
print("\nSaved rocm_smi.txt + versions.txt to", OUT.resolve())


In [ ]:
# --- Cell 2: configuration ---
import os

# Gated Gemma weights need a HF token (accept the model license on huggingface.co first).
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN") or "PASTE_YOUR_HF_TOKEN"
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
assert os.environ["HF_TOKEN"] != "PASTE_YOUR_HF_TOKEN", "Set your Hugging Face token above."

# Attacker + Judge = one strong Gemma (the load-bearing dual-role use). Bump to
# google/gemma-3-27b-it for sharper attacks if your 4-hour budget allows the larger download.
ENGINE_MODEL = "google/gemma-3-12b-it"
ENGINE_PORT  = 8000

# Optional: a deliberately weak Gemma as the system-under-test, for a dramatic ASR.
# Both models run on the one MI300X (192 GB), memory split below. Set False to keep it
# minimal (target = the engine model) and save download budget.
SERVE_WEAK_TARGET = True
TARGET_MODEL = "google/gemma-3-4b-it"
TARGET_PORT  = 8001

# vLLM memory split (fraction of the GPU each server reserves). Sum < 1.0.
ENGINE_MEM_UTIL = 0.55 if SERVE_WEAK_TARGET else 0.90
TARGET_MEM_UTIL = 0.30
MAX_MODEL_LEN   = 4096  # our prompts are short; small KV cache = faster startup

print("engine :", ENGINE_MODEL, "->", f"http://localhost:{ENGINE_PORT}/v1")
print("target :", (TARGET_MODEL if SERVE_WEAK_TARGET else ENGINE_MODEL),
      "->", f"http://localhost:{TARGET_PORT if SERVE_WEAK_TARGET else ENGINE_PORT}/v1")


In [ ]:
# --- Cell 3: launch vLLM server(s) in the background, wait until ready ---
import subprocess, time, urllib.request, shlex

def launch_vllm(model, port, mem_util, logpath):
    cmd = (f"vllm serve {shlex.quote(model)} --port {port} "
           f"--gpu-memory-utilization {mem_util} --max-model-len {MAX_MODEL_LEN} "
           f"--served-model-name {shlex.quote(model)}")
    (OUT / "serve_command.txt").open("a").write(cmd + "\n")
    log = open(logpath, "w")
    proc = subprocess.Popen(shlex.split(cmd), stdout=log, stderr=subprocess.STDOUT)
    return proc

def wait_ready(port, proc, timeout=2400):
    url = f"http://localhost:{port}/v1/models"
    start = time.time()
    while time.time() - start < timeout:
        if proc.poll() is not None:
            raise RuntimeError(f"vLLM on :{port} exited early (code {proc.returncode}); check its log.")
        try:
            urllib.request.urlopen(url, timeout=5)
            print(f"vLLM ready on :{port} after {int(time.time()-start)}s")
            return True
        except Exception:
            time.sleep(5)
    raise TimeoutError(f"vLLM on :{port} not ready within {timeout}s")

engine_proc = launch_vllm(ENGINE_MODEL, ENGINE_PORT, ENGINE_MEM_UTIL, OUT / "vllm_engine.log")
target_proc = None
if SERVE_WEAK_TARGET:
    target_proc = launch_vllm(TARGET_MODEL, TARGET_PORT, TARGET_MEM_UTIL, OUT / "vllm_target.log")

wait_ready(ENGINE_PORT, engine_proc)
if target_proc is not None:
    wait_ready(TARGET_PORT, target_proc)
print("all servers up")


In [ ]:
# --- Cell 4: smoke inference + throughput (proves real generation on MI300X) ---
import time
try:
    from openai import OpenAI
except ImportError:
    subprocess.run(["pip", "-q", "install", "openai>=1.40.0"], check=True)
    from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{ENGINE_PORT}/v1", api_key="EMPTY")

t = time.time()
r = client.chat.completions.create(
    model=ENGINE_MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is the AMD Instinct MI300X?"}],
    max_tokens=128, temperature=0.0,
)
dt = time.time() - t
print(r.choices[0].message.content, "\n")
print("usage:", r.usage, "| latency: %.2fs" % dt)

# rough throughput over a small concurrent batch
import concurrent.futures as cf
prompts = [f"State one true fact about the number {i}." for i in range(8)]
t = time.time()
with cf.ThreadPoolExecutor(max_workers=8) as ex:
    outs = list(ex.map(lambda p: client.chat.completions.create(
        model=ENGINE_MODEL, messages=[{"role": "user", "content": p}], max_tokens=64), prompts))
elapsed = time.time() - t
toks = sum(o.usage.completion_tokens for o in outs)
print("batch of 8: %.2fs | %.1f completion tok/s" % (elapsed, toks / elapsed))
THROUGHPUT = f"{toks/elapsed:.1f} completion tok/s over a batch of 8 ({elapsed:.2f}s)"


In [ ]:
# --- Cell 5: run the FULL GemmaJudge loop against the local MI300X Gemma ---
# Pull the engine code. Uses the feature branch until it is merged to main.
import sys, subprocess, pathlib
REPO_DIR = pathlib.Path("/tmp/gemmajudge")
BRANCH = "feat/backend-engine"   # change to "main" once merged
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH,
                    "https://github.com/Nevern1y/gemmajudge.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
sys.path.insert(0, str(REPO_DIR))
subprocess.run(["pip", "-q", "install", "pydantic>=2.7.0", "openai>=1.40.0"], check=True)

from pydantic import SecretStr
from gemmajudge.config import Settings, EndpointSettings, InferenceBackend, PricingSettings
from gemmajudge.schemas import EvalConfig, FailureMode
from gemmajudge.orchestrator import run_eval

target_base = f"http://localhost:{TARGET_PORT}/v1" if SERVE_WEAK_TARGET else f"http://localhost:{ENGINE_PORT}/v1"
target_model = TARGET_MODEL if SERVE_WEAK_TARGET else ENGINE_MODEL

settings = Settings(
    backend=InferenceBackend.MI300X,
    engine=EndpointSettings(base_url=f"http://localhost:{ENGINE_PORT}/v1",
                            api_key=SecretStr("EMPTY"), model_id=ENGINE_MODEL),
    target=EndpointSettings(base_url=target_base, api_key=SecretStr("EMPTY"), model_id=target_model),
    pricing=PricingSettings(),  # self-hosted: no per-token price to assert
    max_concurrency=8,
)
cfg = EvalConfig(failure_mode=FailureMode.HALLUCINATION, n_cases=10,
                 target_endpoint=target_base, target_model_id=target_model)

# Jupyter has a running event loop -> use top-level await (not asyncio.run()).
result = await run_eval(cfg, settings=settings)

print(f"Attack Success Rate: {result.attack_success_rate:.0%}  "
      f"({sum(1 for v in result.verdicts if v.score>=4)}/{len(result.verdicts)} failed)")
print(f"wall clock: {result.metrics.wall_clock_seconds:.2f}s  |  "
      f"throughput: {result.metrics.throughput_evals_per_sec:.2f} evals/s")
worst = max(result.cases, key=lambda c: c.verdict.score)
print("\nWorst case (drill-down):")
print(" prompt  :", worst.attack.prompt)
print(" response:", worst.verdict.target_response)
print(" score   :", worst.verdict.score, "/5 â€”", worst.verdict.reasoning)

(OUT / "eval_result.json").write_text(result.model_dump_json(indent=2))
print("\nsaved eval_result.json")


In [ ]:
# --- Cell 6: write notes.md (model / VRAM / throughput / timestamp) ---
import datetime

target_desc = TARGET_MODEL if SERVE_WEAK_TARGET else ENGINE_MODEL
mem_desc = str(ENGINE_MEM_UTIL) + (" + " + str(TARGET_MEM_UTIL) if SERVE_WEAK_TARGET else "")
lines = [
    "# AMD MI300X proof - run notes",
    "",
    "- Timestamp (UTC): " + datetime.datetime.utcnow().isoformat() + "Z",
    "- Hardware: AMD Instinct MI300X (see rocm_smi.txt)",
    "- Stack: ROCm 7.2 + vLLM 0.16.0 + PyTorch 2.9 (see versions.txt)",
    "- Attacker+Judge (load-bearing Gemma): " + ENGINE_MODEL,
    "- Target (system-under-test): " + target_desc,
    "- Serving: vLLM OpenAI-compatible, max-model-len " + str(MAX_MODEL_LEN) + ", mem-util " + mem_desc,
    "- Measured throughput: " + THROUGHPUT,
    "- Eval: hallucination, " + str(cfg.n_cases) + " cases, ASR " + format(result.attack_success_rate, ".0%") + " (full run in eval_result.json)",
    "",
    "Gemma runs in both the attacker and judge roles on MI300X - the load-bearing",
    "AMD compute usage for Track 3. Launch command(s) are in serve_command.txt.",
]
notes = chr(10).join(lines)
(OUT / "notes.md").write_text(notes)
print(notes)

## Finish up
1. **Screenshot** the `rocm-smi` cell (Cell 1) and the ASR output (Cell 5) â€” those are the deck visuals.
2. Copy everything from `amd_proof_artifacts/` into the repo at `docs/amd_proof/`, then commit:
   ```bash
   cp amd_proof_artifacts/* /path/to/gemmajudge/docs/amd_proof/
   cd /path/to/gemmajudge && git add docs/amd_proof && git commit -m "Add MI300X proof-of-compute"
   ```
3. Run the teardown cell below to free the GPU and **stop the 4-hour clock**.


In [ ]:
# --- Cell 7: teardown (frees the GPU, stops the budget clock) ---
for name, proc in [("engine", engine_proc), ("target", target_proc)]:
    if proc is not None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except Exception:
            proc.kill()
        print(name, "stopped")
print("GPU released â€” you can sign out.")
